# Device Scan Manual

用来现场验证设备扫描功能，同时也可以对扫到的设备逐台发 *IDN? 确认连通。

建议顺序：
1. 先改参数单元里的 SUBNET
2. 运行参数单元和辅助函数单元
3. 运行扫描单元，确认扫出预期设备
4. 逐台运行 probe 单元确认设备响应正常
5. 运行 YAML 片段单元，把输出粘进 config/devices.local.yaml

In [ ]:
# 参数
SUBNET = "192.168.1"   # 扫描 192.168.1.1-254
TIMEOUT_MS = 1000     # 每个端口探测超时（ms），网络慢时调大
WORKERS = 100         # 并发线程数

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))
from power_control_host.discovery import scan_subnet, devices_to_yaml
print('import ok')

## 1. 扫描局域网

In [ ]:
print(f'Scanning {SUBNET}.1-254 (timeout={TIMEOUT_MS}ms, workers={WORKERS}) ...')
devices = scan_subnet(SUBNET, timeout_ms=TIMEOUT_MS, workers=WORKERS)
if not devices:
    print('No devices found.')
else:
    print(f'Found {len(devices)} device(s):')
    for d in devices:
        print(f'  {d.suggested_id:12}  {d.host}:{d.port}  {d.idn}')

## 2. 逐台 *IDN? 确认

In [ ]:
import socket

def probe_idn(host, port, timeout_ms=3000):
    try:
        with socket.create_connection((host, port), timeout=timeout_ms/1000) as sock:
            sock.settimeout(timeout_ms/1000)
            sock.sendall(b'*IDN?\n')
            resp = b''
            while True:
                chunk = sock.recv(4096)
                if not chunk: break
                resp += chunk
                if b'\n' in resp: break
        return resp.decode(errors='ignore').strip()
    except Exception as e:
        return f'ERROR: {e}'

for d in devices:
    result = probe_idn(d.host, d.port)
    status = 'OK' if 'ERROR' not in result else 'FAIL'
    print(f'[{status}] {d.suggested_id:12}  {d.host}:{d.port}  {result}')

## 3. 统计结果

In [ ]:
odp_list = [d for d in devices if d.vendor == 'owon']
psw_list = [d for d in devices if d.vendor == 'gwinstek']
other_list = [d for d in devices if d.vendor not in ('owon', 'gwinstek')]
print(f'ODP: {len(odp_list)} 台')
print(f'PSW: {len(psw_list)} 台')
if other_list:
    print(f'其他: {len(other_list)} 台')
    for d in other_list:
        print(f'  {d.host}:{d.port}  {d.idn}')

## 4. 输出 YAML 配置片段

把下面输出粘进 `config/devices.local.yaml` 的 `devices:` 列表里。

In [ ]:
if devices:
    print(devices_to_yaml(devices))
else:
    print('没有扫到设备，检查网络连接和子网参数。')

## 5. 单台设备快速验证

改 HOST / PORT 对某台设备做快速冒烟。

In [ ]:
# 改成要验证的设备
TARGET_HOST = devices[0].host if devices else '192.168.1.1'
TARGET_PORT = devices[0].port if devices else 4196

def send_scpi(host, port, command, expect_response=True, timeout_ms=3000):
    print(f'SEND [{host}:{port}]: {command}')
    with socket.create_connection((host, port), timeout=timeout_ms/1000) as sock:
        sock.settimeout(timeout_ms/1000)
        sock.sendall((command + '\n').encode())
        if not expect_response:
            print('RESULT: sent')
            return None
        resp = b''
        while True:
            try:
                chunk = sock.recv(4096)
            except socket.timeout:
                break
            if not chunk: break
            resp += chunk
            if b'\n' in resp: break
    result = resp.decode(errors='ignore').strip()
    print(f'RECV: {result}')
    return result

send_scpi(TARGET_HOST, TARGET_PORT, '*IDN?')

In [ ]:
send_scpi(TARGET_HOST, TARGET_PORT, 'MEAS:VOLT?')

In [ ]:
send_scpi(TARGET_HOST, TARGET_PORT, 'MEAS:CURR?')